In [ ]:
!pip install -q -U "diffusers[torch]" transformers accelerate safetensors matplotlib lpips "pandas<3"
!pip install -q -U --force-reinstall "Pillow<12"
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# 1159_25.png -> seed 1159 -----> laranja ------------- 0
# 1159_29.png -> seed 1159 -----> praia --------------- 1
# 1159_3.png -> seed 1159 ------> ninja --------------- 2
# 1159_7.png -> seed 1159 ------> ourico -------------- 3
# 7836.png -> seed 7836 ---------> astronauta --------- 4
# 9338.png -> seed 9338 -----------> esquilo louco ---- 5

# PICK THE TARGET IMAGE (SEE ABOVE)
TARGET_IMAGE = 5
# NUMBER OF RUNS 
NUM_RUNS = 5

In [ ]:
import gc
import torch

def flush_vram():
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
from pathlib import Path
import zipfile
import urllib.request

MOUNT_GOOGLE_DRIVE = True

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Google Drive mount skipped:", exc)

TARGETS_ZIP_URL = ""

CONTENT_DIR = Path("/content")
DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_PROJECT_DIR = DRIVE_ROOT / "GENAI_TP2"
LOCAL_PROJECT_DIR = CONTENT_DIR / "GENAI_TP2" if CONTENT_DIR.exists() else Path("students")

if DRIVE_ROOT.exists():
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs"
elif CONTENT_DIR.exists():
    OUTPUT_DIR = CONTENT_DIR / "tp2_outputs"
else:
    OUTPUT_DIR = Path("students/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def list_target_images(path):
    path = Path(path)
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
        return [path]
    if not path.exists():
        return []
    return sorted(p for p in path.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)

TARGET_DIR_CANDIDATES = [
    Path("students/tp2-chosen"),
    Path("tp2-chosen"),
    Path("/content/tp2-chosen"),
    Path("/content/tp2_targets"),
    DRIVE_PROJECT_DIR / "tp2-chosen",
    Path("/content/drive/MyDrive/tp2-chosen"),
    Path("/content/drive/MyDrive/tp2_targets"),
]

ZIP_CANDIDATES = [
    Path("students/tp2-chosen.zip"),
    Path("tp2-chosen.zip"),
    Path("/content/tp2-chosen.zip"),
    DRIVE_PROJECT_DIR / "tp2-chosen.zip",
    Path("/content/drive/MyDrive/tp2-chosen.zip"),
]

if TARGETS_ZIP_URL:
    LOCAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    downloaded_zip = LOCAL_PROJECT_DIR / "tp2-chosen.zip"
    urllib.request.urlretrieve(TARGETS_ZIP_URL, downloaded_zip)
    ZIP_CANDIDATES.insert(0, downloaded_zip)
    print("Downloaded targets zip to", downloaded_zip)

if not any(list_target_images(candidate) for candidate in TARGET_DIR_CANDIDATES):
    for zip_path in ZIP_CANDIDATES:
        if zip_path.exists():
            extract_dir = Path("/content/tp2-chosen") if CONTENT_DIR.exists() else Path("tp2-chosen")
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(extract_dir)
            print(f"Extracted {zip_path} -> {extract_dir}")
            break

TARGET_DIR = None
for candidate in TARGET_DIR_CANDIDATES:
    if list_target_images(candidate):
        TARGET_DIR = candidate
        break

if TARGET_DIR is None:
    raise FileNotFoundError(
        "No target images found. Put images in MyDrive/GENAI_TP2/tp2-chosen, "
        "or put tp2-chosen.zip in MyDrive/GENAI_TP2, or set TARGETS_ZIP_URL."
    )

target_images = list_target_images(TARGET_DIR)
print("Target folder:", TARGET_DIR)
print("Output folder:", OUTPUT_DIR)
print("Number of targets:", len(target_images))
target_images

In [ ]:
import csv
import json
import re
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image


def seed_from_filename(path, fallback=2026):
    match = re.match(r"^(\d+)", Path(path).stem)
    return int(match.group(1)) if match else fallback


def safe_stem(path):
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in Path(path).stem)


def load_image(path):
    return Image.open(path).convert("RGB")


def create_run_dir(base_dir=OUTPUT_DIR, identity="student_run"):
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = Path(base_dir) / f"{timestamp}_{identity}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def write_csv(path, rows):
    rows = list(rows)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        Path(path).write_text("")
        return
    fieldnames = []
    for row in rows:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)
    with open(path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def show_images(paths, cols=3, title=None):
    paths = list(paths)
    if not paths:
        print("No images to show.")
        return
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]
    for ax in [ax for row in axes for ax in row]:
        ax.axis("off")
    for ax, path in zip([ax for row in axes for ax in row], paths):
        ax.imshow(load_image(path))
        ax.set_title(Path(path).name)
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


for path in target_images:
    print(Path(path).name, "-> seed", seed_from_filename(path))

In [ ]:
from dataclasses import dataclass
import torch
from diffusers import DiffusionPipeline


@dataclass(frozen=True)
class LCMConfig:
    model_id: str = "SimianLuo/LCM_Dreamshaper_v7"
    seed: int = 2026
    num_inference_steps: int = 8
    guidance_scale: float = 8.0
    lcm_origin_steps: int = 50
    width: int = 768
    height: int = 768


config = LCMConfig()


def default_device():
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = default_device()
print("Using device:", device)


def load_lcm_pipeline(config):
    dtype = torch.float16 if device == "cuda" else torch.float32
    pipe = DiffusionPipeline.from_pretrained(
        config.model_id,
        torch_dtype=dtype,
        use_safetensors=True,
    )
    if hasattr(pipe, "safety_checker"):
        pipe.safety_checker = None
    pipe.to(device)
    return pipe


pipe = load_lcm_pipeline(config)

In [ ]:
def render_prompt(prompt, seed, pipe=pipe, config=config):
    generator_device = "cpu" if device == "mps" else device
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    image = pipe(
        prompt=prompt,
        num_inference_steps=config.num_inference_steps,
        guidance_scale=config.guidance_scale,
        lcm_origin_steps=config.lcm_origin_steps,
        width=config.width,
        height=config.height,
        output_type="pil",
        generator=generator,
    ).images[0]
    return image


def render_prompt_for_target(prompt, target_path):
    seed = seed_from_filename(target_path, config.seed)
    return render_prompt(prompt, seed=seed)


def save_generated_image(image, run_dir, target_path, prompt_index=1):
    target_dir = Path(run_dir) / safe_stem(target_path)
    target_dir.mkdir(parents=True, exist_ok=True)
    path = target_dir / f"candidate_{prompt_index:03d}.png"
    image.save(path)
    return path

In [ ]:
import importlib.util

def import_from_drive(module_name):
    path = f"/content/drive/MyDrive/GENAI_TP2/src/{module_name}.py"
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

In [ ]:
fitness=import_from_drive("fitness")
clip_model, clip_processor = fitness.load_clip(device)
lpips_fn = fitness.load_lpips(device)

target = load_image(target_images[TARGET_IMAGE])
test1 = fitness.compute_fitness(target, target, clip_model, clip_processor, lpips_fn, device)
print(f"Sanity check (target vs target): fitness={test1['fitness']:.4f} clip={test1['clip']:.4f} lpips={test1['lpips']:.4f} rmse={test1['rmse']:.4f}")

In [ ]:

for run_number in range(1, NUM_RUNS + 1):
    print(f"\n\n{'='*80}")
    print(f"RUN {run_number}/{NUM_RUNS}")
    print(f"{'='*80}\n")
    
    RUN_DIR = create_run_dir(OUTPUT_DIR, f"run_{run_number}_TARGET_{TARGET_IMAGE}")
    print(f"Run directory: {RUN_DIR}\n")
    
    VLM_PATH = RUN_DIR / "VLM"
    VLM_PATH.mkdir(parents=True, exist_ok=True)
    candidates_path = VLM_PATH / "vlm_candidates.json"
    vlm_module = import_from_drive("vlm")
    if candidates_path.exists():
        with open(candidates_path, "r") as f:
            data = json.load(f)
        candidates = data["candidates"]
        print(f"Candidates loaded from drive ({len(candidates)})")
    else:
        vlm, vlm_processor = vlm_module.load_vlm()
        candidates = vlm_module.generate_initial_candidates(
            target_path=target_images[TARGET_IMAGE],
            vlm=vlm,
            processor=vlm_processor,
            n_candidates=10,
            temperature=0.9,
        )
        vlm_module.unload_vlm(vlm, vlm_processor)
        del vlm
        del vlm_processor
        flush_vram()
        
        with open(candidates_path, "w") as f:
            json.dump({"target": str(target_images[TARGET_IMAGE]), "candidates": candidates}, f, indent=2)
        print(f"Generated and saved candidates to {candidates_path}")
    
    target = load_image(target_images[TARGET_IMAGE])
    evaluated = []
    VLM_IMAGES = VLM_PATH / "images"
    VLM_IMAGES.mkdir(parents=True, exist_ok=True)
    for i, prompt in enumerate(candidates, 1):
        print(f"[{i:02d}/{len(candidates)}]")
        generated = render_prompt(prompt, seed=seed_from_filename(target_images[TARGET_IMAGE]), pipe=pipe, config=config)
        metrics = fitness.compute_fitness(generated, target, clip_model, clip_processor, lpips_fn, device)
        evaluated.append({"prompt": prompt, "generated": generated, **metrics})
        print(f"fitness={metrics['fitness']:.4f} clip={metrics['clip']:.4f} lpips={metrics['lpips']:.4f} rmse={metrics['rmse']:.4f}")
        generated.save(VLM_IMAGES / f"generated_{i:03d}.png")
    
    opro_module = import_from_drive("OPRO")
    
    OPRO_PATH = RUN_DIR / "OPRO"
    OPRO_PATH.mkdir(parents=True, exist_ok=True)
    OPRO_IMAGES = OPRO_PATH / "images"
    OPRO_IMAGES.mkdir(parents=True, exist_ok=True)
    OPRO_CHECKPOINTS = OPRO_PATH / "checkpoints"
    OPRO_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    
    checkpoints = sorted(OPRO_CHECKPOINTS.glob("opro_iter_*.json"))
    
    if checkpoints:
        with open(checkpoints[-1], "r") as f:
            population = json.load(f)
        best_fitness = max(c["fitness"] for c in population)
        no_improve_count = 0
        iteration = population[0].get("iteration", 0)
        print(f" Checkpoint loaded — iteration {iteration}, best fitness {best_fitness:.4f}")
    else:
        population = [
            {"prompt": c["prompt"], "fitness": c["fitness"],
             "clip": c["clip"], "lpips": c["lpips"], "rmse": c["rmse"]}
            for c in evaluated
        ]
        best_fitness = max(c["fitness"] for c in population)
        no_improve_count = 0
        iteration = 0
        print(f" Starting OPRO from iteration 0 : {len(population)} candidates")
    
    llm, processor = opro_module.load_llm()
    
    WARMUP_ITERATIONS = 5
    while True:
        iteration += 1
        print(f"\n[Iteration {iteration}]")
    
        new_prompts = opro_module.generate_initial_candidates(
            target_images[TARGET_IMAGE], llm, processor, population, clip_model, clip_processor, n_candidates=5
        )
    
        new_candidates = []
        for prompt in new_prompts:
            if not opro_module.is_diverse_enough(prompt, population, clip_model, clip_processor):
                print(f" skipped to similar {prompt}")
                continue
            generated = render_prompt(prompt, seed=seed_from_filename(target_images[TARGET_IMAGE]), pipe=pipe, config=config)
            metrics = fitness.compute_fitness(generated, target, clip_model, clip_processor, lpips_fn, device)
            new_candidates.append({
                "prompt": prompt,
                "fitness": metrics["fitness"],
                "clip": metrics["clip"],
                "lpips": metrics["lpips"],
                "rmse": metrics["rmse"],
                "iteration": iteration,
            })
            print(f"  fitness={metrics['fitness']:.4f} | {prompt}")
    
        population = sorted(population + new_candidates, key=lambda x: x["fitness"], reverse=True)[:20]
    
        current_best = population[0]["fitness"]
        avg_fitness = sum(c["fitness"] for c in population) / len(population)
        print(f" Best: {current_best:.4f} | Mean: {avg_fitness:.4f}")
    
        checkpoint_path = OPRO_CHECKPOINTS / f"opro_iter_{iteration:03d}.json"
        with open(checkpoint_path, "w") as f:
            json.dump(population, f, indent=2)
        print(f"  Checkpoint saved: {checkpoint_path.name}")
    
        if iteration > WARMUP_ITERATIONS:
            if current_best > best_fitness:
                best_fitness = current_best
                no_improve_count = 0
            else:
                no_improve_count += 1
                print(f" No improvement ({no_improve_count}/5)")
            best_image = render_prompt(population[0]["prompt"], seed=seed_from_filename(target_images[TARGET_IMAGE]), pipe=pipe, config=config)
            best_image.save(OPRO_IMAGES / f"best_iter_{iteration:03d}.png")
    
            if no_improve_count >= 5:
                print(f" 5 iterations without improvement.")
                break
        else:
            if current_best > best_fitness:
                best_fitness = current_best
            print(f" Warmup iteration {iteration}/{WARMUP_ITERATIONS}")
    
    opro_module.unload_llm(llm, processor)
    del llm
    del processor
    flush_vram()
    
    print(f"\n OPRO terminates — best fitness: {population[0]['fitness']:.4f}")
    print(f" {population[0]['prompt']}")
    
    ga_module = import_from_drive("ga")
    
    GA_PATH = RUN_DIR / "GA"
    GA_PATH.mkdir(parents=True, exist_ok=True)
    GA_IMAGES = GA_PATH / "images"
    GA_IMAGES.mkdir(parents=True, exist_ok=True)
    GA_CHECKPOINTS = GA_PATH / "checkpoints"
    GA_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    
    ga_checkpoints = sorted(GA_CHECKPOINTS.glob("ga_iter_*.json"))
    if ga_checkpoints:
        with open(ga_checkpoints[-1], "r") as f:
            ga_population = json.load(f)
        ga_iteration = ga_population[0].get("ga_iteration", 0)
        ga_best_fitness = max(c["fitness"] for c in ga_population)
        print(f"GA checkpoint loaded — iteration {ga_iteration}, best fitness {ga_best_fitness:.4f}")
    else:
        opro_checkpoints = sorted(OPRO_CHECKPOINTS.glob("opro_iter_*.json"))
        if not opro_checkpoints:
            raise FileNotFoundError("No OPRO checkpoints found. Run OPRO first.")
        with open(opro_checkpoints[-1], "r") as f:
            ga_population = json.load(f)
        ga_iteration = 0
        ga_best_fitness = max(c["fitness"] for c in ga_population)
        print(f"GA seeded from {opro_checkpoints[-1].name} — {len(ga_population)} candidates, best fitness {ga_best_fitness:.4f}")
    
    print(f"Done — {len(ga_population)} candidates ready.")
    
    ga_llm, ga_processor = ga_module.load_llm()
    
    GA_MAX_ITERATIONS = 50
    
    for _ in range(GA_MAX_ITERATIONS):
        ga_iteration += 1
        threshold = ga_module.DIVERSITY_THRESHOLD + 0.15 * (ga_iteration / GA_MAX_ITERATIONS)
        print(f"\n[GA Iteration {ga_iteration}] threshold={threshold:.3f}")
    
        new_prompts = ga_module.evolve(
            ga_llm, ga_processor, ga_population,
            clip_model, clip_processor,
            n_candidates=5,
            threshold=threshold,
        )
    
        new_candidates = []
        for j, prompt in enumerate(new_prompts):
            generated = render_prompt(prompt, seed=seed_from_filename(target_images[TARGET_IMAGE]), pipe=pipe, config=config)
            metrics = fitness.compute_fitness(generated, target, clip_model, clip_processor, lpips_fn, device)
            img_path = GA_IMAGES / f"cand_iter_{ga_iteration:03d}_{j:02d}.png"
            generated.save(img_path)
            new_candidates.append({
                "prompt": prompt,
                "fitness": metrics["fitness"],
                "clip": metrics["clip"],
                "lpips": metrics["lpips"],
                "rmse": metrics["rmse"],
                "ga_iteration": ga_iteration,
            })
            print(f"  fitness={metrics['fitness']:.4f} | {prompt}")
    
        ga_population = sorted(ga_population + new_candidates, key=lambda x: x["fitness"], reverse=True)[:20]
    
        current_best = ga_population[0]["fitness"]
        avg_fitness = sum(c["fitness"] for c in ga_population) / len(ga_population)
        print(f" Best: {current_best:.4f} | Mean: {avg_fitness:.4f}")
    
        checkpoint_path = GA_CHECKPOINTS / f"ga_iter_{ga_iteration:03d}.json"
        with open(checkpoint_path, "w") as f:
            json.dump(ga_population, f, indent=2)
        print(f"  Checkpoint saved: {checkpoint_path.name}")
    
        best_image = render_prompt(ga_population[0]["prompt"], seed=seed_from_filename(target_images[TARGET_IMAGE]), pipe=pipe, config=config)
        best_image.save(GA_IMAGES / f"best_iter_{ga_iteration:03d}.png")
    
    ga_module.unload_llm(ga_llm, ga_processor)
    del ga_llm
    del ga_processor
    flush_vram()
    
    print(f"\n GA terminates — best fitness: {ga_population[0]['fitness']:.4f}")
    print(f" {ga_population[0]['prompt']}")
    print(f"\nRun {run_number} completed. Results saved to {RUN_DIR}")

print(f"\n\nAll {NUM_RUNS} runs completed!")

In [ ]:
import os
import time
print("Waiting 5 seconds to end connection with server (saving resources).")
time.sleep(5)
os._exit(0)